In [19]:
import torch 
print("PyTorch Version:", torch.__version__)

print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)

PyTorch Version: 2.6.0+cu124
CUDA Available: True
CUDA Version: 12.4


In [1]:
#loading the Dataset and DATAlOADER


import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.utils import save_image
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import glob
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

IMG_SIZE = (512, 512)  
BATCH_SIZE = 8 
original_path = "dataset_originals-20250317T080721Z-001/dataset_originals"
distorted_path = "dataset_distorted-20250317T080722Z-001/dataset_distorted"

if not os.path.exists(original_path) or not os.path.exists(distorted_path):
    raise FileNotFoundError("⚠️ ERROR: Dataset paths do not exist! Check paths.")

original_images = sorted(glob.glob(os.path.join(original_path, "*.jpg")) + 
                         glob.glob(os.path.join(original_path, "*.png")) + 
                         glob.glob(os.path.join(original_path, "*.jpeg")))

distorted_images = sorted(glob.glob(os.path.join(distorted_path, "*.jpg")) + 
                          glob.glob(os.path.join(distorted_path, "*.png")) + 
                          glob.glob(os.path.join(distorted_path, "*.jpeg")))

# ✅ LIMIT to 10K image pairs
original_images = original_images[:10000]
distorted_images = distorted_images[:10000]

print(f"✅ Found {len(original_images)} original images")
print(f"✅ Found {len(distorted_images)} distorted images")


# ============================
# 📌 Image Dataset Class
# ============================
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),  # Converts to [0,1] range
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize to [-1,1]
])


✅ Found 10000 original images
✅ Found 10000 distorted images


In [2]:
#Transforming the images as and creating the dataset 

class ImageDataset(Dataset):
    def __init__(self, distorted_list, original_list, transform=None):
        self.distorted_list = distorted_list
        self.original_list = original_list
        self.transform = transform
    
    def __len__(self):
        return len(self.distorted_list)
    
    def __getitem__(self, index):
        distorted_img = Image.open(self.distorted_list[index]).convert("RGB")
        original_img = Image.open(self.original_list[index]).convert("RGB")
        
        if self.transform:
            distorted_img = self.transform(distorted_img)
            original_img = self.transform(original_img)
        
        return distorted_img, original_img

# Create dataset & dataloader
dataset = ImageDataset(distorted_images, original_images, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=6, pin_memory=True)

print("✅ Dataset and DataLoader created successfully!")

✅ Dataset and DataLoader created successfully!


In [3]:
#Create Generator model 

import torch
import torch.nn as nn
import torch.optim as optim

class UNetGenerator(nn.Module):
    def __init__(self):
        super(UNetGenerator, self).__init__()

        def down_block(in_channels, out_channels, normalize=True):
            layers = [nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.LeakyReLU(0.2))
            return nn.Sequential(*layers)

        def up_block(in_channels, out_channels, dropout=0.0):
            layers = [
                nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU()
            ]
            if dropout:
                layers.append(nn.Dropout(dropout))
            return nn.Sequential(*layers)

        # Encoder (Downsampling)
        self.down1 = down_block(3, 64, normalize=False)
        self.down2 = down_block(64, 128)
        self.down3 = down_block(128, 256)
        self.down4 = down_block(256, 512)
        self.down5 = down_block(512, 512)

        # Decoder (Upsampling)
        self.up1 = up_block(512, 512, dropout=0.5)
        self.up2 = up_block(512, 256, dropout=0.5)
        self.up3 = up_block(256, 128)
        self.up4 = up_block(128, 64)
        self.final = nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)

        u1 = self.up1(d5)
        u2 = self.up2(u1 + d4)
        u3 = self.up3(u2 + d3)
        u4 = self.up4(u3 + d2)
        out = torch.tanh(self.final(u4 + d1))
        return out


In [4]:
import torch
import torch.nn as nn

class PatchDiscriminator(nn.Module):
    def __init__(self):
        super(PatchDiscriminator, self).__init__()

        def discriminator_block(in_channels, out_channels, normalization=True):
            layers = [nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1)]  # Fixed padding
            if normalization:
                layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)

        self.model = nn.Sequential(
            discriminator_block(6, 64, normalization=False),  # Input: (512x512) -> (256x256)
            discriminator_block(64, 128),  # (256x256) -> (128x128)
            discriminator_block(128, 256),  # (128x128) -> (64x64)
            discriminator_block(256, 512),  # (64x64) -> (32x32)
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)  # (32x32) -> (30x30)
        )

    def forward(self, x, y):
        input_data = torch.cat((x, y), dim=1)  # Concatenate input and target images
        return self.model(input_data)


In [5]:
lambda_l1 = 100  # Weight for L1 loss

adv_loss = nn.BCEWithLogitsLoss()
l1_loss = nn.L1Loss()


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import time
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

# Initialize models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = UNetGenerator().to(device)
discriminator = PatchDiscriminator().to(device)

# Optimizers
optimizerG = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerD = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Loss functions
lambda_l1 = 100  # Weight for L1 loss
adv_loss = nn.BCEWithLogitsLoss()
l1_loss = nn.L1Loss()

# FID & Inception Score Metrics
fid = FrechetInceptionDistance(normalize=True).to(device)
inception = InceptionScore().to(device)

# Results storage
results = []

# Training loop
EPOCHS = 250 # Set number of epochs
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    print(f"🔄 Starting Epoch {epoch}/{EPOCHS}...")  

    epoch_d_loss = 0
    epoch_g_loss = 0

    for distorted_images, original_images in dataloader:
        distorted_images, original_images = distorted_images.to(device), original_images.to(device)

        # Train Discriminator
        optimizerD.zero_grad()
        real_output = discriminator(original_images, distorted_images)
        real_labels = torch.ones_like(real_output)

        fake_images = generator(distorted_images).detach()
        fake_output = discriminator(fake_images, distorted_images)
        fake_labels = torch.zeros_like(fake_output)

        real_loss = adv_loss(real_output, real_labels)
        fake_loss = adv_loss(fake_output, fake_labels)

        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizerD.step()
        torch.cuda.empty_cache()

        # Train Generator
        optimizerG.zero_grad()
        generated_images = generator(distorted_images)
        fake_output = discriminator(generated_images, distorted_images)

        g_adv_loss = adv_loss(fake_output, real_labels)
        g_l1_loss = l1_loss(generated_images, original_images)
        g_loss = g_adv_loss + lambda_l1 * g_l1_loss

        g_loss.backward()
        optimizerG.step()

        # Accumulate loss
        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()

# ✅ Now delete to save memory
        del real_output, fake_output, real_loss, fake_loss, d_loss, g_adv_loss, g_l1_loss, g_loss
        torch.cuda.empty_cache()


    # Compute FID and Inception Score at the end of the epoch
    with torch.no_grad():
        fid.update(generated_images, real=False)
        fid.update(original_images, real=True)
        fid_score = fid.compute().item()
        fid.reset()

        generated_images_uint8 = ((generated_images + 1) * 127.5).clamp(0, 255).to(torch.uint8)
        inception.update(generated_images_uint8)
        inception_score, _ = inception.compute()
        inception_score = inception_score.item()
        inception.reset()

    # Average loss
    avg_d_loss = epoch_d_loss / len(dataloader)
    avg_g_loss = epoch_g_loss / len(dataloader)

    # ETA
    elapsed_time = time.time() - start_time
    eta = elapsed_time * (EPOCHS - epoch)
    eta_mins, eta_secs = divmod(int(eta), 60)

    print(f"✅ Epoch {epoch}/{EPOCHS} - D Loss: {avg_d_loss:.4f} - G Loss: {avg_g_loss:.4f} - FID: {fid_score:.4f} - Inception Score: {inception_score:.4f}")
    print(f"⏳ Estimated Time Remaining: {eta_mins} min {eta_secs} sec")

    # Save results
    results.append([epoch, avg_d_loss, avg_g_loss, fid_score, inception_score])
    df = pd.DataFrame(results, columns=["Epoch", "D Loss", "G Loss", "FID", "Inception Score"])
    df.to_csv("results.csv", index=False)

    # Save latest
    torch.save(generator.state_dict(), "generator_latest.pth")
    torch.save(discriminator.state_dict(), "discriminator_latest.pth")

    # Optional: save every 10 epochs
    if epoch % 10 == 0:
        torch.save(generator.state_dict(), f"generator_epoch{epoch}.pth")
        torch.save(discriminator.state_dict(), f"discriminator_epoch{epoch}.pth")


/home/student/anaconda3/envs/pytorch/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


🔄 Starting Epoch 1/250...
✅ Epoch 1/250 - D Loss: 0.4075 - G Loss: 17.1680 - FID: 271.6389 - Inception Score: 1.0000
⏳ Estimated Time Remaining: 1387 min 21 sec
🔄 Starting Epoch 2/250...
✅ Epoch 2/250 - D Loss: 0.4627 - G Loss: 15.9820 - FID: 198.4860 - Inception Score: 1.0000
⏳ Estimated Time Remaining: 1365 min 0 sec
🔄 Starting Epoch 3/250...
✅ Epoch 3/250 - D Loss: 0.4657 - G Loss: 15.4016 - FID: 207.4330 - Inception Score: 1.0000
⏳ Estimated Time Remaining: 1332 min 8 sec
🔄 Starting Epoch 4/250...
✅ Epoch 4/250 - D Loss: 0.4454 - G Loss: 15.0069 - FID: 191.2457 - Inception Score: 1.0000
⏳ Estimated Time Remaining: 1350 min 50 sec
🔄 Starting Epoch 5/250...
✅ Epoch 5/250 - D Loss: 0.4281 - G Loss: 14.8181 - FID: 224.6401 - Inception Score: 1.0000
⏳ Estimated Time Remaining: 1297 min 52 sec
🔄 Starting Epoch 6/250...
✅ Epoch 6/250 - D Loss: 0.4360 - G Loss: 14.5450 - FID: 272.9736 - Inception Score: 1.0000
⏳ Estimated Time Remaining: 1279 min 2 sec
🔄 Starting Epoch 7/250...
✅ Epoch 7/2

In [13]:
original_path = "dataset_originals-20250317T080721Z-001/dataset_originals"
distorted_path = "dataset_distorted-20250317T080722Z-001/dataset_distorted"

original_images = sorted(glob.glob(os.path.join(original_path, "*.jpg")) + 
                         glob.glob(os.path.join(original_path, "*.png")) + 
                         glob.glob(os.path.join(original_path, "*.jpeg")))

distorted_images = sorted(glob.glob(os.path.join(distorted_path, "*.jpg")) + 
                          glob.glob(os.path.join(distorted_path, "*.png")) + 
                          glob.glob(os.path.join(distorted_path, "*.jpeg")))

original_images = original_images[:10000]
distorted_images = distorted_images[:10000]

print(f"Originals: {len(original_images)} | Distorted: {len(distorted_images)}")


Originals: 10000 | Distorted: 10000


In [14]:
dataset = ImageDataset(distorted_images, original_images, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=6, pin_memory=True)


In [16]:
for i in range(5):
    print(os.path.basename(distorted_images[i]), "<-->", os.path.basename(original_images[i]))


000038527b455eaccd15e623f2e229ecdbceba2b.jpg <--> 000038527b455eaccd15e623f2e229ecdbceba2b.jpg
00012e7f370296610a118b1a939906f8011c1a6d.jpg <--> 00012e7f370296610a118b1a939906f8011c1a6d.jpg
000408a4d4f3563abf4c3fe16cef57d0ec4922a5.jpg <--> 000408a4d4f3563abf4c3fe16cef57d0ec4922a5.jpg
00067aee03b47f0757897c774b36e66e4d126a00.jpg <--> 00067aee03b47f0757897c774b36e66e4d126a00.jpg
0006e7ed3650794b844a976d39b11069f9eee7a6.jpg <--> 0006e7ed3650794b844a976d39b11069f9eee7a6.jpg


In [ ]:
import time
import torch

# === Load model weights (already done above) ===
# === Optimizers and losses (already defined above) ===

for epoch in range(start_epoch, total_epochs + 1):
    start_time = time.time()
    running_loss_G = 0.0
    running_loss_D = 0.0
    total_batches = len(dataloader)

    for i, batch in enumerate(dataloader):
        real_A = batch[0].to(device)
        real_B = batch[1].to(device)

        # === Train Generator ===
        optimizer_G.zero_grad()
        fake_B = generator(real_A)
        pred_fake = discriminator(real_A, fake_B)
        loss_G_GAN = criterion_GAN(pred_fake, torch.ones_like(pred_fake))
        loss_G_L1 = criterion_L1(fake_B, real_B) * lambda_L1
        loss_G = loss_G_GAN + loss_G_L1
        loss_G.backward()
        optimizer_G.step()

        # === Train Discriminator ===
        optimizer_D.zero_grad()
        with torch.no_grad():
            fake_B_detached = fake_B.detach()
        pred_real = discriminator(real_A, real_B)
        pred_fake = discriminator(real_A, fake_B_detached)
        loss_D_real = criterion_GAN(pred_real, torch.ones_like(pred_real))
        loss_D_fake = criterion_GAN(pred_fake, torch.zeros_like(pred_fake))
        loss_D = (loss_D_real + loss_D_fake) * 0.5
        loss_D.backward()
        optimizer_D.step()

        running_loss_G += loss_G.item()
        running_loss_D += loss_D.item()

        # Clear intermediate tensors
        del real_A, real_B, fake_B, pred_fake, pred_real, fake_B_detached
        torch.cuda.empty_cache()

        print(f"Epoch [{epoch}/{total_epochs}] | Batch [{i+1}/{total_batches}] | "
              f"Loss_G: {loss_G.item():.4f} | Loss_D: {loss_D.item():.4f}", end='\r')

    # === End of epoch summary ===
    avg_loss_G = running_loss_G / total_batches
    avg_loss_D = running_loss_D / total_batches
    epoch_time = time.time() - start_time
    eta = (total_epochs - epoch) * epoch_time / 60  # ETA in minutes

    print(f"\n✅ Epoch [{epoch}/{total_epochs}] done in {epoch_time:.2f}s | "
          f"Avg Loss_G: {avg_loss_G:.4f} | Avg Loss_D: {avg_loss_D:.4f} | ETA: {eta:.1f} min")

    if epoch % 100 == 0:
        torch.save(generator.state_dict(), f'generator_epoch{epoch}.pth')
        torch.save(discriminator.state_dict(), f'discriminator_epoch{epoch}.pth')

    # Extra memory cleanup at epoch end
    torch.cuda.empty_cache()


Epoch [251/1000] | Batch [63/1250] | Loss_G: 13.7213 | Loss_D: 0.0543